# Battery-run output aggregation

This notebook combines schema-v2 `battery_results_*.csv` files, adds run identifiers and configuration metadata, and writes the aggregated dataset used by the analysis notebooks.


In [ ]:

from pathlib import Path
import json
import pandas as pd
import re
import sys

import matplotlib.pyplot as plt
import seaborn as sns

# The working directory of the kernel can be either the repo root or the notebook folder.
# We select the root robustly by looking for tests/battery/results.
candidate_root = Path().resolve()
if (candidate_root / "tests" / "battery" / "results").exists():
    REPO_ROOT = candidate_root
else:
    # The notebook is under tests/battery: move up to the repository root
    REPO_ROOT = candidate_root.parents[1]

RESULTS_DIR = REPO_ROOT / "tests" / "battery" / "results"
sys.path.insert(0, str(RESULTS_DIR.parent))

# Get folders that look like runs
# prefix_regex = r"^(cf|cfg)\d+_"
# get only cfg runs
# prefix_regex = r"^(cfg)[0-5]+_071[8|7]"
# prefix_regex = r"^(cfg)_onlyhigh_[0-5]+_"
# prefix_regex = r"^(cfg)_onlyhigh_[0-5]+_0719_16"
# prefix_regex = r"^(cfg)_onlyhigh_[a-zA-Z_]+_[0-5]+_07"
# prefix_regex = r"^(cfg)_onlyhigh_[a-zA-Z_]+_[0-1]+_07(22|23)"
# prefix_regex = r"^(cfg)_onlyhigh_[A-Za-z_]+_[01]+_07(?:22_(?:18|19|20|21)|23)$"

# prefix_regex = (
#     r"^cfg_onlyhigh_.*_(?:"
#     r"0722_(?:18|19|20|21|22|23)\d{2}"
#     r"|"
#     r"0723_\d{4}"
#     r")_"
# )
# prefix_regex = r"^(cfg_onlyhigh_dp_0_0725_1721_8_roll0_0_F_M|cfg_onlyhigh_singleton_best_0_0725_1735_8_roll0_0_T_M|cfg_onlyhigh_singleton_best_0_0725_1727_8_roll0_0_T_M|cfg_onlyhigh_dp_0_0725_1739_8_roll0_0_F_M)"

prefix_regex = r"^(cfg)_test([1|2|3|4|5|6])_"

run_dirs = sorted(
    d for d in RESULTS_DIR.iterdir()
    if d.is_dir() and re.match(prefix_regex, d.name)
)
print(f"Trovate {len(run_dirs)} run directories")
for d in run_dirs:
    print(" -", d.name)


In [ ]:
from nb_utils import require_output_schema
from save_results import aggregate_and_save

aggregate_and_save(run_dirs=run_dirs)
output_csv = RESULTS_DIR / "aggregated_battery_results.csv"
require_output_schema(output_csv, schema_name="carbonshift.battery-aggregate")
df_all = pd.read_csv(output_csv)


In [ ]:
pd.set_option("display.max_columns", None)
display(df_all.head())
pd.set_option("display.max_rows", None)
display(
    df_all.sort_values(
        by=["scenario_id", "strategy", "batch_size", "final_global_error", "parallelism"]
    )
)


In [ ]:
df_batch_size_1 = df_all[df_all["batch_size"].isin([1, 4])]
display(
    df_batch_size_1[
        ["scenario_id", "strategy", "batch_size", "carbon_saving", "final_global_error", "reqs_fallback_assigned"]
    ].sort_values(
        by=["strategy", "scenario_id", "batch_size", "carbon_saving", "final_global_error"]
    )
)


In [ ]:
display(df_all["reqs_fallback_assigned"].describe())
display(
    df_all.sort_values(
        by=["reqs_fallback_assigned", "carbon_saving"], ascending=[False, False]
    ).head(10)
)


# Analysis section

This is a section I used to get some statistics and plots from the aggregated data with the purpose of gathering insights on the performance of the different configurations. 

In [ ]:
display(df_all["scenario_id"].value_counts())
display(df_all["strategy"].value_counts())
display(df_all["reqs_fallback_assigned"].value_counts())


In [ ]:
df_all_sorted_by_scenario = df_all.sort_values(by=["scenario_id", "carbon_cost"])
df_top5_per_scenario = df_all_sorted_by_scenario.groupby("scenario_id").head(5)
pd.set_option("display.max_rows", 200)
display(df_top5_per_scenario)

scenario = "high_load_1500_92"
df_scenario = df_all_sorted_by_scenario[df_all_sorted_by_scenario["scenario_id"] == scenario]
display(df_scenario.sort_values(by=["strategy", "batch_size", "carbon_cost"]))


In [ ]:
numeric_cols = df_all.select_dtypes(include=["number"]).columns
df_dp = df_all[df_all["strategy"] == "dp"]
if not df_dp.empty:
    corr_matrix = df_dp[numeric_cols].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f")


## Carryover vs dp

In [ ]:

# Count dp vs dp in the top 5 per scenario
count_modes = df_top5_per_scenario["strategy"].value_counts()
print(count_modes)

# Avg elapsed_seconds per mode 
# get rows where strategy does not include "offline"
df_online_modes = df_all[~df_all["strategy"].str.contains("offline")]
avg_elapsed_seconds_per_mode = df_online_modes.groupby("strategy")["elapsed_seconds"].mean()
print(avg_elapsed_seconds_per_mode)
avg_elapsed_seconds_per_mode = df_top5_per_scenario.groupby("strategy")["elapsed_seconds"].mean()
print(avg_elapsed_seconds_per_mode)

# Avg carbon cost per mode
avg_carbon_cost_per_mode = df_online_modes.groupby("strategy")["carbon_cost"].mean()
print(avg_carbon_cost_per_mode)



In [ ]:
# install seaborn
# !pip install seaborn

###

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt


# df_gr_ret = df_all[~df_all["strategy"].str.contains("offline|online")]
# plt.figure(figsize=(10,10))


# sns.boxplot(
#     data=df_gr_ret,
#     x="scenario_id",
#     y="carbon_saving",
#     hue="strategy",
#     showfliers=True,
# )

# plt.xlabel("Scenario ID")
# plt.ylabel("Carbon Saving")
# plt.title("Carbon Saving by Scenario and Infeasibility Mode")
# plt.xticks(rotation=90)
# plt.tight_layout()
# # plt.show()

## Carryover vs dp
Overall dp is in general faster and yields the same results as dp.\
However, considering only top 5 runs per scenario, both the carbon cost and the elapsed time are basically the same.\
In general, it could be better to use dp, as it is faster and yields the same results.

These tests were run with the following `dp` parameters:
```
infeasibility_recovery_mode: "dp_last_slot".to_string(),
infeasibility_mock_influence: 0.6,
infeasibility_mock_influence_decay_step: 0.15,
```
That is, in case of infeasibility, the algorithm will consider the previous slot's assignment with a weight of 0.6 to influence the current slot's assignment. The weight will decay by 0.15 at every subsequent slot where infeasibility occurs. This means that the algorithm will try to "carry over" the previous slot's assignment to the current slot, but with decreasing influence as more infeasibility occurs.\
The mechanism is designed to help the solver not to "panic" and try to assign requests knowing that others will come, allowing to mitigate the error average.

This probably results in being only partially effective due to the decayed window, which kind of embeds this idea already.


## Parallelism and throughput


In [ ]:
# Compare elapsed time and carbon saving for DP runs at different worker parallelism.
df_dp = df_all[df_all["strategy"] == "dp"]
group_cols = ["scenario_id", "batch_size"]

baseline = (
    df_dp[df_dp["parallelism"] == 1]
    .groupby(group_cols)
    .agg(elapsed_parallelism_1=("elapsed_seconds", "max"))
    .reset_index()
)
parallel = df_dp[df_dp["parallelism"] > 1][
    group_cols
    + [
        "parallelism",
        "elapsed_seconds",
        "peak_concurrent_workers",
        "avg_concurrent_workers",
        "carbon_saving",
        "reqs_fallback_assigned",
    ]
]
comparison = parallel.merge(baseline, on=group_cols, how="inner")
comparison["seconds_saved"] = comparison["elapsed_parallelism_1"] - comparison["elapsed_seconds"]
comparison["speedup_pct"] = comparison["seconds_saved"] / comparison["elapsed_parallelism_1"] * 100
comparison = comparison[
    [
        "scenario_id",
        "batch_size",
        "parallelism",
        "elapsed_seconds",
        "elapsed_parallelism_1",
        "seconds_saved",
        "speedup_pct",
        "carbon_saving",
        "peak_concurrent_workers",
        "avg_concurrent_workers",
        "reqs_fallback_assigned",
    ]
]
display(comparison.head())

summary = comparison.groupby("parallelism").agg(
    runs=("scenario_id", "count"),
    avg_saved_seconds=("seconds_saved", "mean"),
    median_saved_seconds=("seconds_saved", "median"),
    avg_speedup_pct=("speedup_pct", "mean"),
    median_speedup_pct=("speedup_pct", "median"),
    avg_peak_workers=("peak_concurrent_workers", "mean"),
    avg_workers=("avg_concurrent_workers", "mean"),
)
summary


Increasing `max_batch_solver_parallelism` can reduce wall-clock time when enough batch work is available.

Assignments are priced against live slot occupancy at commit time, so measurements can vary with machine contention. Compare repeated runs together with worker-utilization metrics.

In [ ]:
display(df_top5_per_scenario["reqs_fallback_assigned"].value_counts())
display(df_top5_per_scenario["parallelism"].value_counts())


Non incredibilmente, sembra che i migliori risultati si ottengano con parallelismo a 1, non c'è concorrenza e non c'è rischio di fare assegnamenti basati su informazioni stale, potenzialmente sforando un cap level e avendo un costo superiore a quello previsto.

In [ ]:
# get top5 but filter out parallelism = 1
df_all_sorted_by_scenario_filtered = df_all_sorted_by_scenario[df_all_sorted_by_scenario["parallelism"] != 1]
df_top5_per_scenario = df_all_sorted_by_scenario_filtered.groupby("scenario_id").head(5)

df_top5_per_scenario

In [ ]:
# Compare average and maximum carbon savings across the available strategies.
average_carbon_saving = df_all.groupby("strategy")["carbon_saving"].mean()
display(average_carbon_saving)

online_aco = df_all[df_all["strategy"] == "online_ant_colony"]
display(online_aco["carbon_saving"].max())

## Quali combinazioni di parametri portano a un carbon saving maggiore?

This analysis is limited to `strategy == "dp"` and examines `parallelism`, `batch_size`, `reqs_in`, `reqs_out`, and `reqs_fallback_assigned`.

In [ ]:
FEATURES = [
    "parallelism",
    "batch_size",
    "reqs_in",
    "reqs_out",
    "reqs_fallback_assigned",
    "avg_concurrent_workers",
]
TARGET = "carbon_saving"
df_gf_raw = df_all[(df_all["strategy"] == "dp") & (df_all["parallelism"] > 1)].copy()
df_gf = df_gf_raw.dropna(subset=FEATURES + [TARGET]).copy()
print(f"Rows in the DP parallelism analysis: {len(df_gf)}")
df_gf[FEATURES + [TARGET]].describe()


### Correlazione con `carbon_saving`

Prima approssimazione: correlazione lineare (Pearson) di ciascun campo con `carbon_saving`. Valori positivi indicano che crescere quel parametro tende ad aumentare il risparmio di carbonio.

In [ ]:
corr_with_target = df_gf[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

plt.figure(figsize=(7, 4))
colors = ["tab:red" if v < 0 else "tab:green" for v in corr_with_target.values]
corr_with_target.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlazione di Pearson con carbon_saving")
plt.title("Correlazione lineare parametri → carbon_saving (greedy_fallback)")
plt.tight_layout()
plt.show()


### Vista multidimensionale (PCA)

Le 5 variabili vengono standardizzate e proiettate in 2 componenti principali; ogni punto è colorato in base al `carbon_saving` osservato. Le frecce mostrano il contributo (loading) di ciascun campo alle due componenti: punti/regioni che si trovano nella stessa direzione di una freccia tendono ad avere valori più alti di quel campo. Questo aiuta a individuare visivamente quali *combinazioni* di parametri (non solo singoli valori) sono associate a un carbon saving più alto.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df_gf[FEATURES])
pca = PCA(n_components=2)
scores = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    scores[:, 0], scores[:, 1],
    c=df_gf[TARGET], cmap="viridis", s=40, alpha=0.85, edgecolor="none",
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("carbon_saving (%)")

# Biplot: draw feature loadings as arrows scaled to the score range for readability.
loadings = pca.components_.T  # shape (n_features, 2)
scale = np.abs(scores).max() * 1.15
for i, feat in enumerate(FEATURES):
    ax.annotate(
        "", xy=(loadings[i, 0] * scale, loadings[i, 1] * scale), xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="firebrick", lw=1.8),
    )
    ax.text(
        loadings[i, 0] * scale * 1.12, loadings[i, 1] * scale * 1.12, feat,
        color="firebrick", fontsize=10, ha="center", va="center", weight="bold",
        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.75),
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)")
ax.set_title("PCA dei parametri (greedy_fallback), colore = carbon_saving", pad=20)
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
ax.margins(0.3)  # extra room so loading labels don't get clipped/overlap the frame or title
plt.tight_layout()
plt.show()


### `batch_size` × `parallelism`: carbon saving medio

Heatmap del `carbon_saving` medio per ogni combinazione di `batch_size` e `parallelism` (sempre limitata a `greedy_fallback`); il numero in ogni cella è il numero di run osservate.

In [ ]:
pivot_mean = df_gf.pivot_table(
    index="parallelism", columns="batch_size", values=TARGET, aggfunc="mean"
).sort_index(ascending=False)
pivot_count = df_gf.pivot_table(
    index="parallelism", columns="batch_size", values=TARGET, aggfunc="count"
).sort_index(ascending=False)

annot = pivot_mean.round(1).astype(str) + "\n(n=" + pivot_count.fillna(0).astype(int).astype(str) + ")"

plt.figure(figsize=(1.3 * len(pivot_mean.columns) + 2, 1.1 * len(pivot_mean.index) + 2))
sns.heatmap(
    pivot_mean, annot=annot, fmt="", cmap="RdYlGn", cbar_kws={"label": "carbon_saving (%) medio"},
)
plt.xlabel("batch_size")
plt.ylabel("parallelism")
plt.title("Carbon saving medio per batch_size × parallelism (greedy_fallback)")
plt.tight_layout()
plt.show()


In [ ]:
pivot_mean = df_gf.pivot_table(
    index="parallelism", columns="batch_size", values=TARGET, aggfunc="mean"
).sort_index(ascending=False)
pivot_count = df_gf.pivot_table(
    index="parallelism", columns="batch_size", values=TARGET, aggfunc="count"
).sort_index(ascending=False)

annot = pivot_mean.round(1).astype(str) + "\n(n=" + pivot_count.fillna(0).astype(int).astype(str) + ")"

plt.figure(figsize=(1.3 * len(pivot_mean.columns) + 2, 1.1 * len(pivot_mean.index) + 2))
sns.heatmap(
    pivot_mean, annot=annot, fmt="", cmap="RdYlGn", cbar_kws={"label": "carbon_saving (%) medio"},
)
plt.xlabel("batch_size")
plt.ylabel("parallelism")
plt.title("Mean carbon saving by batch size and parallelism (DP)")
plt.tight_layout()
plt.show()


In [ ]:
# discretize avg_concurrent_workers into 10 bins, the first 9 must represent values in 0-1, 1-2, 2-3, ... and the last one will be 10+
# Work on a copy so we don't overwrite the numeric column in df_gf (reused by later cells).
df_workers_binned = df_gf.copy()
df_workers_binned["avg_concurrent_workers"] = pd.cut(
    df_workers_binned["avg_concurrent_workers"],
    bins=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, np.inf],
    labels=["0-1", "1-2", "2-3", "3-4", "4-5", "5-6", "6-7", "7-8", "8-9", "10+"],
)

pivot_mean = df_workers_binned.pivot_table(
    index="avg_concurrent_workers", columns="batch_size", values=TARGET, aggfunc="mean", observed=True,
).sort_index(ascending=False)
pivot_count = df_workers_binned.pivot_table(
    index="avg_concurrent_workers", columns="batch_size", values=TARGET, aggfunc="count", observed=True,
).reindex(index=pivot_mean.index, columns=pivot_mean.columns)

# n indicates the number of runs for each cell in the heatmap
annot = pivot_mean.round(1).astype(str) + "\n(n=" + pivot_count.fillna(0).astype(int).astype(str) + ")"

plt.figure(figsize=(1.3 * len(pivot_mean.columns) + 2, 1.1 * len(pivot_mean.index) + 2))
sns.heatmap(
    pivot_mean, annot=annot, fmt="", cmap="RdYlGn", cbar_kws={"label": "carbon_saving (%) medio"},
)
plt.xlabel("batch_size")
plt.ylabel("avg_concurrent_workers")
plt.title("Carbon saving medio per batch_size × avg_concurrent_workers (greedy_fallback)")
plt.tight_layout()
plt.show()


In [ ]:
pivot_mean = df_gf.pivot_table(
    index="reqs_in", columns="batch_size", values=TARGET, aggfunc="mean"
).sort_index(ascending=False)
pivot_count = df_gf.pivot_table(
    index="reqs_in", columns="batch_size", values=TARGET, aggfunc="count"
).sort_index(ascending=False)

annot = pivot_mean.round(1).astype(str) + "\n(n=" + pivot_count.fillna(0).astype(int).astype(str) + ")"

plt.figure(figsize=(1.3 * len(pivot_mean.columns) + 2, 1.1 * len(pivot_mean.index) + 2))
sns.heatmap(
    pivot_mean, annot=annot, fmt="", cmap="RdYlGn", cbar_kws={"label": "carbon_saving (%) medio"},
)
plt.xlabel("batch_size")
plt.ylabel("reqs_in")
plt.title("Carbon saving medio per batch_size × reqs_in (greedy_fallback)")
plt.tight_layout()
plt.show()


In [ ]:
# get avg carbon saving for strategy = greedy_fallback and batch_size = 1
df_gf_parallelism_1 = df_all[
    (df_all["strategy"] == "dp") &
    (df_all["batch_size"] == 1)
]
# get avg carbon saving for strategy = offline_singleton_best and whatever batch_size
df_gc_parallelism_1 = df_all[
    (df_all["strategy"] == "offline_singleton_best")]

display(df_gf_parallelism_1.groupby("batch_size")["carbon_saving"].mean())
# get avg carbon saving for strategy = offline_singleton_best and whatever batch_size
display(df_gc_parallelism_1["carbon_saving"].mean())